In [1]:

# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc
from dash import html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
# Using the AnimalShelter class developed in Project One.
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################

# FIX ME update with your username and password and CRUD Python module name
# The database credentials are required for the course environment.
# In a production application, credentials should use environment variables.
username = "aacuser"
password = "NicoMiso00$$"

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'], inplace=True, errors='ignore')

# Make sure the initial database query returned data.
if df.empty:
    raise ValueError("No animal records were returned from MongoDB.")

# Convert MongoDB-specific values into table-friendly values.
# Keep latitude, longitude, and age numeric for filtering and mapping.
def prepare_dataframe(records):
    """Convert MongoDB records into a dashboard-ready DataFrame."""
    data = pd.DataFrame.from_records(records)

    # Remove MongoDB document IDs before displaying records.
    data.drop(columns=['_id'], inplace=True, errors='ignore')

    # Convert age and location fields to numeric values.
    numeric_columns = [
        'age_upon_outcome_in_weeks',
        'location_lat',
        'location_long'
    ]

    for column in numeric_columns:
        if column in data.columns:
            data[column] = pd.to_numeric(
                data[column], errors='coerce'
            )

    # Replace missing values with None for Dash compatibility.
    data = data.astype(object).where(pd.notnull(data), None)

    return data


df = prepare_dataframe(df.to_dict('records'))

# Maintain a consistent set of columns for all filtered results.
dashboard_columns = [
    {"name": column, "id": column,
     "deletable": False, "selectable": True}
    for column in df.columns
]

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


##########################################
# Rescue Type Database Queries / Model
##########################################

# These filters follow the Grazioso Salvare Dashboard Specifications.
# Each query retrieves dogs meeting the required breed, sex, and age.

# Water Rescue:
# Labrador Retriever Mix, Chesapeake Bay Retriever, Newfoundland
# Intact Female, between 26 and 156 weeks old.
water_rescue_query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "Labrador Retriever Mix",
            "Chesapeake Bay Retriever",
            "Newfoundland"
        ]
    },
    "sex_upon_outcome": "Intact Female",
    "age_upon_outcome_in_weeks": {
        "$gte": 26,
        "$lte": 156
    }
}

# Mountain or Wilderness Rescue:
# German Shepherd, Alaskan Malamute, Old English Sheepdog,
# Siberian Husky, and Rottweiler.
# Intact Male, between 26 and 156 weeks old.
mountain_rescue_query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "German Shepherd",
            "Alaskan Malamute",
            "Old English Sheepdog",
            "Siberian Husky",
            "Rottweiler"
        ]
    },
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {
        "$gte": 26,
        "$lte": 156
    }
}

# Disaster Rescue or Individual Tracking:
# Doberman Pinscher, German Shepherd, Golden Retriever,
# Bloodhound, and Rottweiler.
# Intact Male, between 20 and 300 weeks old.
disaster_rescue_query = {
    "animal_type": "Dog",
    "breed": {
        "$in": [
            "Doberman Pinscher",
            "German Shepherd",
            "Golden Retriever",
            "Bloodhound",
            "Rottweiler"
        ]
    },
    "sex_upon_outcome": "Intact Male",
    "age_upon_outcome_in_weeks": {
        "$gte": 20,
        "$lte": 300
    }
}

# Associate each interactive filter with its MongoDB query.
rescue_queries = {
    "Water Rescue": water_rescue_query,
    "Mountain or Wilderness Rescue": mountain_rescue_query,
    "Disaster or Individual Tracking": disaster_rescue_query,
    "Reset": {}
}


#########################
# Dashboard Layout / View
#########################

app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
# The logo must be stored in the same folder as this notebook.
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image

# Encode the logo so it can be displayed inside the dashboard.
with open(image_filename, 'rb') as image_file:
    encoded_image = base64.b64encode(image_file.read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

# Define colors used throughout the dashboard.
primary_color = '#9D0039'
background_color = '#F5F6F8'
border_color = '#DDDDDD'

# Set up the main dashboard layout.
app.layout = html.Div(
    style={
        'fontFamily': 'Arial, sans-serif',
        'backgroundColor': background_color,
        'padding': '20px',
        'minHeight': '100vh'
    },
    children=[

        # Dashboard header with the client logo and developer information.
        html.Div(
            style={
                'textAlign': 'center',
                'backgroundColor': 'white',
                'padding': '20px',
                'borderRadius': '8px'
            },
            children=[

                # Grazioso Salvare logo linked to the client homepage.
                html.A(
                    href='https://www.snhu.edu',
                    target='_blank',
                    children=[
                        html.Img(
                            src='data:image/png;base64,{}'.format(
                                encoded_image.decode()
                            ),
                            style={
                                'height': '150px',
                                'maxWidth': '100%',
                                'objectFit': 'contain'
                            }
                        )
                    ]
                ),

                html.Center(
                    html.B(
                        html.H1(
                            'CS-340 Grazioso Salvare Dashboard',
                            style={'color': primary_color}
                        )
                    )
                ),

                # Unique identifier required by the client.
                html.P(
                    'Developed by Michael Donaldson',
                    style={
                        'fontSize': '16px',
                        'color': '#555555'
                    }
                ),

                html.P(
                    'Austin Animal Center Search and Rescue Dashboard',
                    style={'color': '#666666'}
                )
            ]
        ),

        html.Hr(),

        # Interactive rescue type filtering section.
        html.Div(
            style={
                'backgroundColor': 'white',
                'padding': '20px',
                'borderRadius': '8px',
                'marginBottom': '15px'
            },
            children=[

                html.H3(
                    'Rescue Type Filters',
                    style={'color': primary_color}
                ),

                html.P(
                    'Select a rescue type to view dogs that meet '
                    'the required training criteria.'
                ),

                #FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.

                # Radio items allow only one filter to be active at a time.
                dcc.RadioItems(
                    id='filter-type',
                    options=[
                        {
                            'label': 'Water Rescue',
                            'value': 'Water Rescue'
                        },
                        {
                            'label': 'Mountain or Wilderness Rescue',
                            'value': 'Mountain or Wilderness Rescue'
                        },
                        {
                            'label': 'Disaster or Individual Tracking',
                            'value': 'Disaster or Individual Tracking'
                        },
                        {
                            'label': 'Reset',
                            'value': 'Reset'
                        }
                    ],
                    value='Reset',
                    labelStyle={
                        'display': 'inline-block',
                        'marginRight': '25px',
                        'marginBottom': '10px',
                        'cursor': 'pointer'
                    },
                    inputStyle={'marginRight': '7px'}
                )
            ]
        ),

        html.Hr(),

        # Interactive data table displaying animal records.
        html.Div(
            style={
                'backgroundColor': 'white',
                'padding': '15px',
                'borderRadius': '8px'
            },
            children=[

                html.H3(
                    'Austin Animal Center Outcomes',
                    style={'color': primary_color}
                ),

                dash_table.DataTable(
                    id='datatable-id',

                    columns=dashboard_columns,
                    data=df.to_dict('records'),

                    #FIXME: Set up the features for your interactive data table to make it user-friendly for your client
                    #If you completed the Module Six Assignment, you can copy in the code you created here

                    # Enable pagination to prevent displaying every record.
                    page_size=10,
                    page_action='native',

                    # Allow users to sort and filter records.
                    sort_action='native',
                    filter_action='native',

                    # Allow users to select one animal.
                    row_selectable='single',
                    selected_rows=[0],

                    # Allow users to highlight table columns.
                    column_selectable='multi',
                    selected_columns=[],

                    # Keep the table readable with many columns.
                    style_table={
                        'overflowX': 'auto',
                        'maxHeight': '450px',
                        'overflowY': 'auto'
                    },

                    style_cell={
                        'textAlign': 'left',
                        'padding': '10px',
                        'minWidth': '120px',
                        'width': '150px',
                        'maxWidth': '220px',
                        'overflow': 'hidden',
                        'textOverflow': 'ellipsis'
                    },

                    style_header={
                        'backgroundColor': primary_color,
                        'color': 'white',
                        'fontWeight': 'bold'
                    },

                    style_data={
                        'backgroundColor': 'white',
                        'color': '#222222'
                    },

                    style_data_conditional=[
                        {
                            'if': {'row_index': 'odd'},
                            'backgroundColor': '#F3F3F3'
                        }
                    ],

                    tooltip_delay=0,
                    tooltip_duration=None
                )
            ]
        ),

        html.Br(),
        html.Hr(),

        #This sets up the dashboard so that your chart and your geolocation chart are side-by-side
        html.Div(
            className='row',
            style={
                'display': 'flex',
                'flexWrap': 'wrap',
                'gap': '20px'
            },
            children=[

                # Breed distribution chart.
                html.Div(
                    id='graph-id',
                    className='col s12 m6',
                    style={
                        'flex': '1 1 400px',
                        'minWidth': '0',
                        'backgroundColor': 'white',
                        'padding': '15px',
                        'borderRadius': '8px'
                    }
                ),

                # Animal geolocation map.
                html.Div(
                    id='map-id',
                    className='col s12 m6',
                    style={
                        'flex': '1 1 400px',
                        'minWidth': '0',
                        'backgroundColor': 'white',
                        'padding': '15px',
                        'borderRadius': '8px'
                    }
                )
            ]
        )
    ]
)


#############################################
# Interaction Between Components / Controller
#############################################

# Update the dashboard data based on the selected rescue type.
@app.callback(
    [Output('datatable-id', 'data'),
     Output('datatable-id', 'selected_rows')],
    [Input('filter-type', 'value')]
)
def update_dashboard(filter_type):

    ## FIX ME Add code to filter interactive data table with MongoDB queries
    #
    #
    #        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    #        data=df.to_dict('records')
    #
    #
    #        return (data,columns)

    # Retrieve the MongoDB query for the selected rescue type.
    query = rescue_queries.get(filter_type, {})

    # The CRUD read method returns matching animal records.
    records = db.read(query)

    # Convert the records into a DataFrame.
    filtered_df = prepare_dataframe(records)

    # An empty result should clear the table.
    if filtered_df.empty:
        return [], []

    # Return records and automatically select the first row.
    return filtered_df.to_dict('records'), [0]


# Display the breeds of animal based on quantity represented in
# the data table
@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")]
)
def update_graphs(viewData):

    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #

    #return [
    #    dcc.Graph(
    #        figure = px.pie(df, names='breed', title='Preferred Animals')
    #    )
    #]

    # Use the data currently visible after sorting and filtering.
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)

    # Display a message when the current filter has no results.
    if dff.empty or 'breed' not in dff.columns:
        return html.Div([
            html.H3('Breed Distribution'),
            html.P('No animal records match the current filters.')
        ])

    # Count how many animals belong to each breed.
    breed_counts = (
        dff['breed']
        .fillna('Unknown')
        .value_counts()
        .reset_index()
    )

    breed_counts.columns = ['breed', 'count']

    # Display the ten most common breeds.
    # Group the remaining breeds as Other for readability.
    top_breeds = breed_counts.head(10).copy()

    if len(breed_counts) > 10:
        other_count = breed_counts.iloc[10:]['count'].sum()

        top_breeds = pd.concat([
            top_breeds,
            pd.DataFrame([{
                'breed': 'Other',
                'count': other_count
            }])
        ], ignore_index=True)

    # Create the pie chart.
    fig = px.pie(
        top_breeds,
        names='breed',
        values='count',
        title='Animal Breed Distribution',
        hole=0.25
    )

    fig.update_layout(
        margin=dict(l=10, r=10, t=60, b=10),
        legend_title_text='Breed'
    )

    return [
        dcc.Graph(
            figure=fig,
            style={'width': '100%'}
        )
    ]


#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):

    # Preserve alternating row colors and highlight selected columns.
    styles = [{
        'if': {'row_index': 'odd'},
        'backgroundColor': '#F3F3F3'
    }]

    styles.extend([{
        'if': {'column_id': i},
        'background_color': '#D2F3FF'
    } for i in (selected_columns or [])])

    return styles


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")]
)
def update_map(viewData, index):

    # Use the original records when no table transformation exists.
    if viewData is None:
        dff = df.copy()
    else:
        dff = pd.DataFrame.from_records(viewData)

    # Show a message if no animals match the selected filter.
    if dff.empty:
        return html.Div([
            html.H3('Animal Geolocation'),
            html.P('No animal records are available to map.')
        ])

    # Because we only allow single row selection, the list can be converted to a row index here
    if not index:
        row = 0
    else:
        row = index[0]

    # Prevent invalid selections after filtering or sorting.
    if row < 0 or row >= len(dff):
        row = 0

    # Retrieve the selected animal using column names.
    # This is safer than relying on fixed column positions.
    animal = dff.iloc[row]

    latitude = pd.to_numeric(
        animal.get('location_lat'), errors='coerce'
    )

    longitude = pd.to_numeric(
        animal.get('location_long'), errors='coerce'
    )

    # Austin TX is at [30.75,-97.48]
    default_center = [30.75, -97.48]

    # Validate coordinates before creating a map marker.
    has_location = (
        pd.notna(latitude)
        and pd.notna(longitude)
        and -90 <= latitude <= 90
        and -180 <= longitude <= 180
    )

    # Only draw a marker when coordinates are valid.
    if has_location:
        center = [float(latitude), float(longitude)]
    else:
        center = default_center

    # Read the animal name and breed for the map popup.
    animal_name = animal.get('name') or 'Unknown'
    animal_breed = animal.get('breed') or 'Unknown'

    # Create the map's required tile layer.
    map_children = [
        dl.TileLayer(id='base-layer-id')
    ]

    # Marker with tool tip and popup
    # Column 13 and 14 define the grid-coordinates for the map
    # Column 4 defines the breed for the animal
    # Column 9 defines the name of the animal

    # Use named columns rather than fixed indexes to avoid errors
    # if the data table's column order changes.
    if has_location:
        map_children.append(
            dl.Marker(
                position=center,
                children=[
                    dl.Tooltip(str(animal_breed)),
                    dl.Popup([
                        html.H3('Animal Name'),
                        html.P(str(animal_name)),
                        html.H4('Breed'),
                        html.P(str(animal_breed))
                    ])
                ]
            )
        )

    # Display the map and related animal information.
    return [
        html.H3(
            'Animal Geolocation',
            style={'color': primary_color}
        ),

        html.P(
            'Selected Animal: {}'.format(animal_name)
        ),

        dl.Map(
            style={
                'width': '100%',
                'height': '450px'
            },
            center=center,
            zoom=10,
            children=map_children
        ),

        html.P(
            'No valid location is available for this animal.'
            if not has_location else
            'Select another animal in the table to update the map.',
            style={
                'fontSize': '13px',
                'color': '#555555',
                'marginTop': '10px'
            }
        )
    ]


##############################
# Dashboard Application Launch
##############################

# Start the dashboard in the Codio Jupyter environment.
app.run_server(debug=True, host='0.0.0.0', port=8050)


Dash app running on http://0.0.0.0:8050/
